# Session 9 — Loss Functions & Output Heads

Runs Part 1 harness, Part 2 MTP smoke, Part 3 Curriculum_Required_stats plots top-to-bottom.

**Local:** open from `Assign_009_Init/` so `src` imports resolve, then run all cells.

**Colab:** upload a complete zip (must include `web/` + `widgets/`), run the Colab setup cell, then remaining cells.
Use `requirements.txt` only on Colab (not `requirements-jupyter.txt`).

**Part 3:** Curriculum_Required_stats PNGs under `reports/` (peak VRAM, chunk frontier, SwiGLU).


## Colab setup (skip on local Jupyter)

1. Zip `Assign_009_Init` with at least: `src/`, `data/`, `tests/`, `web/`, `widgets/`, `notebooks/`, `requirements.txt` (omit `.venv/`). **DIST needs `web/` + `widgets/`.**
2. Upload zip to Colab (Files ↑ or `files.upload()`).
3. Run the next cell. It installs **only** `requirements.txt` (no Jupyter pins — Colab already has Jupyter).
   Do **not** install `requirements-jupyter.txt` on Colab.

Faster Colab (skip static site): `run_all.main(["--skip-dist"])`.


In [ ]:
# === COLAB ONLY — skip this cell when running locally ===
import os
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

REQUIRED_REL = [
    "src/pipeline/run_all.py",
    "web/index.html",
    "widgets/s9_boot.js",
    "tests/build_dist.py",
    "data/raw/shakespeare_tiny.txt",
    "requirements.txt",
]


def _find_project_root() -> Path:
    """Prefer /content/Assign_009_Init; also accept flat /content if src+web live there."""
    candidates = [Path("/content/Assign_009_Init"), Path("/content")]
    # Also scan one level under /content
    content = Path("/content")
    if content.is_dir():
        for p in sorted(content.iterdir()):
            if p.is_dir() and p.name not in {".config", "sample_data"}:
                candidates.append(p)
    for c in candidates:
        if (c / "src").is_dir() and (c / "web" / "index.html").is_file():
            return c
    for c in candidates:
        if (c / "src").is_dir():
            return c
    return Path("/content/Assign_009_Init")


if IN_COLAB:
    ZIP_NAME = "Assign_009_Init.zip"
    zip_path = Path("/content") / ZIP_NAME
    if zip_path.is_file():
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall("/content")
    target = _find_project_root()
    os.chdir(target)
    if str(target) not in sys.path:
        sys.path.insert(0, str(target))
    missing = [r for r in REQUIRED_REL if not (target / r).is_file()]
    if missing:
        raise SystemExit(
            "Colab project tree incomplete under "
            + str(target)
            + ". Missing:\n  - "
            + "\n  - ".join(missing)
            + "\nRe-zip Assign_009_Init including web/ and widgets/, re-upload, Runtime > Restart."
        )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )
    print("Colab cwd=", Path.cwd())
    print("ROOT ok; web/ and widgets/ present.")
else:
    print("Not Colab — skip unzip/pip; use local .venv")


In [1]:
import os
import sys
from pathlib import Path

# Resolve project root (folder containing src/ and data/)
cwd = Path.cwd()
root = cwd if (cwd / "src").exists() else cwd.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("cwd=", Path.cwd())

cwd= D:\A1_School_ai_25\ERA_V5\Assign_009_Init


In [ ]:
# Full graded pipeline: Part 1 + Part 2 + Part 3 + widget export + dist/
# Reload so Jupyter does not keep a stale run_all.main from an earlier import.
# Pass [] so Jupyter's -f kernel-....json is not fed to argparse.
# Faster Colab (optional): main(["--skip-dist", "--skip-widget"])
import importlib
import src.pipeline.run_all as run_all
importlib.reload(run_all)
run_all.main([])


In [ ]:
from pathlib import Path

print(Path("data/evaluation/seven_numbers.json").read_text()[:800])
print("---")
print(Path("data/evaluation/session09_writeup.txt").read_text())
pngs = sorted(Path("reports").glob("Curriculum_Required_stats_*.png"))
print("Part3 PNGs:", [str(p).replace("\\", "/") for p in pngs])
assert len(pngs) >= 3, "expected three Curriculum_Required_stats PNGs"